In [1]:
import cv2
import os
import time

# =========================
# KONFIGURASI
# =========================
DATASET_PATH = "dataset_angka_video_tes"

START_LABEL = 1
END_LABEL = 30

VIDEOS_PER_LABEL = 50
VIDEO_DURATION = 3
FPS = 20

COUNTDOWN = 3
CAMERA_INDEX = 0

# =========================
# SETUP DATASET
# =========================
LABELS = [str(i) for i in range(START_LABEL, END_LABEL + 1)]

os.makedirs(DATASET_PATH, exist_ok=True)

for label in LABELS:
    os.makedirs(os.path.join(DATASET_PATH, label), exist_ok=True)

# =========================
# BUKA KAMERA
# =========================
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    raise RuntimeError("Kamera tidak terbuka. Coba ganti CAMERA_INDEX ke 1.")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

stop_program = False

print("Kontrol:")
print("S = mulai rekam")
print("N = lanjut/skip ke angka berikutnya")
print("Q = keluar")

# =========================
# FUNGSI TAMPILKAN TEKS
# =========================
def put_text(frame, text, y, color=(255, 255, 255), scale=0.8, thickness=2):
    cv2.putText(
        frame,
        text,
        (30, y),
        cv2.FONT_HERSHEY_SIMPLEX,
        scale,
        color,
        thickness
    )

# =========================
# LOOP UTAMA
# =========================
try:
    for label in LABELS:
        if stop_program:
            break

        print(f"\n=== Siap merekam angka {label} ===")

        label_path = os.path.join(DATASET_PATH, label)

        existing_videos = [
            f for f in os.listdir(label_path)
            if f.lower().endswith((".mp4", ".avi", ".mov"))
        ]

        count = len(existing_videos)

        while count < VIDEOS_PER_LABEL:
            ret, frame = cap.read()
            if not ret:
                print("Frame kamera tidak terbaca.")
                stop_program = True
                break

            frame = cv2.flip(frame, 1)
            display = frame.copy()

            put_text(display, f"Label angka: {label}", 40, (0, 255, 0), 1)
            put_text(display, f"Video ke: {count + 1}/{VIDEOS_PER_LABEL}", 80)
            put_text(display, "S: rekam | N: skip label | Q: keluar", 120, (0, 255, 255), 0.7)

            cv2.imshow("Record Dataset Angka", display)

            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                stop_program = True
                break

            elif key == ord("n"):
                print(f"Skip angka {label}.")
                break

            elif key == ord("s"):
                # Countdown sebelum rekam
                for sec in range(COUNTDOWN, 0, -1):
                    start_countdown = time.time()

                    while time.time() - start_countdown < 1:
                        ret, frame = cap.read()
                        if not ret:
                            break

                        frame = cv2.flip(frame, 1)
                        countdown_frame = frame.copy()

                        put_text(countdown_frame, f"Siap angka {label}", 40, (0, 255, 0), 1)
                        put_text(countdown_frame, f"Mulai dalam {sec}...", 90, (0, 0, 255), 1.2)
                        put_text(countdown_frame, "Posisikan tangan dari awal gerakan", 140, (0, 255, 255), 0.7)

                        cv2.imshow("Record Dataset Angka", countdown_frame)

                        if cv2.waitKey(1) & 0xFF == ord("q"):
                            stop_program = True
                            break

                    if stop_program:
                        break

                if stop_program:
                    break

                filename = os.path.join(
                    label_path,
                    f"{label}_{count + 1:03d}.mp4"
                )

                out = cv2.VideoWriter(
                    filename,
                    fourcc,
                    FPS,
                    (frame_width, frame_height)
                )

                print(f"Merekam: {filename}")

                start_time = time.time()

                while time.time() - start_time < VIDEO_DURATION:
                    ret, frame = cap.read()
                    if not ret:
                        print("Frame kamera tidak terbaca saat rekam.")
                        break

                    frame = cv2.flip(frame, 1)

                    elapsed = time.time() - start_time
                    remaining = max(0, VIDEO_DURATION - elapsed)

                    record_frame = frame.copy()

                    put_text(record_frame, f"RECORDING angka {label}", 40, (0, 0, 255), 1)
                    put_text(record_frame, f"Sisa: {remaining:.1f} detik", 80, (0, 0, 255), 0.8)
                    put_text(record_frame, f"Video: {count + 1}/{VIDEOS_PER_LABEL}", 120, (255, 255, 255), 0.7)

                    out.write(frame)
                    cv2.imshow("Record Dataset Angka", record_frame)

                    if cv2.waitKey(1) & 0xFF == ord("q"):
                        stop_program = True
                        break 

                out.release()

                if stop_program:
                    break

                count += 1
                print(f"Selesai video {count}/{VIDEOS_PER_LABEL} untuk angka {label}")

        print(f"Label angka {label} selesai.")

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Program selesai dengan aman.")

Kontrol:
S = mulai rekam
N = lanjut/skip ke angka berikutnya
Q = keluar

=== Siap merekam angka 1 ===
Merekam: dataset_angka_video_tes\1\1_001.mp4
Selesai video 1/50 untuk angka 1
Merekam: dataset_angka_video_tes\1\1_002.mp4
Selesai video 2/50 untuk angka 1
Label angka 1 selesai.
Program selesai dengan aman.


In [10]:
import cv2
import os
import time

FOLDER = "photo"
TOTAL_GAMBAR = 60
DELAY = 5

os.makedirs(FOLDER, exist_ok=True)

cap = cv2.VideoCapture(0)

count = len([f for f in os.listdir(FOLDER) if f.lower().endswith(".jpg")])
pending_capture = False
start_time = 0

while True:
    ret, frame = cap.read()

    if not ret:
        print("Kamera gagal dibuka")
        break

    frame = cv2.flip(frame, 1)

    # Frame bersih untuk disimpan
    clean_frame = frame.copy()

    # Frame tampilan boleh ada teks
    display_frame = frame.copy()

    if pending_capture:
        elapsed = time.time() - start_time
        sisa = int(DELAY - elapsed) + 1

        cv2.putText(display_frame, f"Capture dalam: {max(sisa, 0)} detik",
                    (10, 120), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0, 0, 255), 2)

        if elapsed >= DELAY:
            count += 1

            filename = os.path.join(FOLDER, f"photo_{count}.jpg")
            cv2.imwrite(filename, clean_frame)

            print(f"Foto {count} tersimpan: {filename}")

            pending_capture = False

            if count >= TOTAL_GAMBAR:
                print("Selesai capture 30 foto")
                break

    cv2.putText(display_frame, f"Foto: {count}/{TOTAL_GAMBAR}",
                (10, 40), cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2)

    cv2.putText(display_frame, "Tekan C = hitung 5 detik | Q = keluar",
                (10, 80), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255, 255, 255), 2)

    cv2.imshow("Manual Capture Delay", display_frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('c'):
        if not pending_capture and count < TOTAL_GAMBAR:
            pending_capture = True
            start_time = time.time()
            print("Siap-siap, foto diambil 5 detik lagi...")

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Siap-siap, foto diambil 5 detik lagi...
Foto 48 tersimpan: photo\photo_48.jpg
Siap-siap, foto diambil 5 detik lagi...
Foto 49 tersimpan: photo\photo_49.jpg
Siap-siap, foto diambil 5 detik lagi...
Foto 50 tersimpan: photo\photo_50.jpg
Siap-siap, foto diambil 5 detik lagi...
Foto 51 tersimpan: photo\photo_51.jpg
Siap-siap, foto diambil 5 detik lagi...
Foto 52 tersimpan: photo\photo_52.jpg
